1. 프롬프팅 시뮬레이션 — Zero-shot / Few-shot / CoT / ReAct 비교
2. ReActStep / ReActTrace 클래스 — ReAct 루프 구조 시뮬레이션
3. ReActPromptTemplate 클래스 — 프롬프트 자동 생성기
4. ReActParser 클래스 — 정규표현식 기반 출력 파서

In [3]:
# ========================================
# 실제로 보내는 프롬프트 문자열 (어떤 형식인지)
# =========================================
question = '대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?'

# ① Zero-shot
zero_shot_prompt = f'''질문:{question}\n답변:'''

# ② Few-shot
few_shot_prompt = f'''질문:일본의 수도는 어디이며, 그 도시의 인구는 얼마인가?
답변: 일본의 수도는 도쿄이며,인구는 약 1,500만 명입니다.
질문:{question}\n답변:'''

# ③ Chain-of-Thought
cot_prompt = f'''질문:{question}\n단계별로 생각해보겠습니다.\n1단계: 대한민국의 수도를 확인합니다.
2단계: 해당 도시의 인구 데이터를 조회합니다.\n답변:'''

# ④ ReAct
react_prompt = f'''질문:{question}
Thought: ...
Action: Search[대한민국수도]
Observation: 서울특별시입니다.
...
Action: Finish[...]
'''

In [2]:
class ReActStep:
    """ReAct의 단일 단계(Thought-Action-Orservation)을 표현"""
    def __init__(self, step_num, thought, action, action_input, observation):
        self.step_num = step_num    # 몇 번째 단계인지
        self.thought = thought      # LLM이 이 단계에서 한 추론 내용
        self.action = action        # 사용할 도구 이름
        self.action_input = action_input    # 도구 실행 후 반환 결과
        self.observation = observation

    def display(self):
        # 이 단계의 내용을 보기 좋게 출력하는 메소드
        # 도구이름[입력값] 형태로 출력
        print(f'-- step {self.step_num} --')
        print(f'Thought {self.thought}')
        print(f'Action {self.action}[{self.action_input}]')  # ReAct 표준 형식
        print(f'Observation {self.observation}')
        print() 

In [1]:
class ReActTrace:
    """ReAct의 전체 추론 과정(trace)을 관리하는 클래스"""

    def __init__(self, question):
        self.question = question    # 풀어야 할 원래 질문
        self.steps = []
        self.final_answer = None    # finish() 호출 전까지 None 유지

    def add_step(self, thought, action, action_input, observation):
        # ReActStep 객체를 새로 만들어서 self.steps 리스트에 추가
        step = ReActStep(len(self.steps) + 1, thought, action, action_input, observation)
        self.steps.append(step)

        return self     # 매서드 체이닝을 위해 자기 자신을 반환

    def finish(self, answer):
        self.final_answer = answer  # 최종 답변 저장
        return self

    def display(self):
        print(f"Question: {self.question}")  # 원래 질문 먼저 출력

        for step in self.steps:     # 저장된 ReActStep 객체들을 순서대로 순회
            step.display()          # 각 단계의 display 호출

        if self.final_answer:
            print(f"Final Answer: {self.final_answer}")
            

In [4]:
trace = ReActTrace("2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?")
trace.add_step(
    thought="이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.",
    action="Search",                         # 사용할 도구: 검색
    action_input="2024년 노벨 물리학상 수상자",  # 검색어
    observation="존 홉필드와 제프리 힌턴이 수상했습니다."  # 검색 결과(시뮬레이션)
)
# → 1번 스텝 생성 후 리스트에 추가

trace.add_step(
    thought="수상자를 확인했습니다. 이제 연구 분야를 조회해야 합니다.",
    action="Search",
    action_input="존 홉필드 제프리 힌턴 연구 분야",
    observation="홉필드 네트워크, 볼츠만 머신, 역전파 알고리즘 등 인공 신경망 개척."
)
# → 2번 스텝 생성 후 리스트에 추가

trace.finish("존 홉필드와 제프리 힌턴, 주요 분야는 인공 신경망과 기계 학습의 기초 이론")
# → final_answer 저장

trace.display()
# → Question 출력 → 1번 스텝 → 2번 스텝 → Final Answer 순서로 출력

Question: 2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?
-- step 1 --
Thought 이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.
Action Search[2024년 노벨 물리학상 수상자]
Observation 존 홉필드와 제프리 힌턴이 수상했습니다.

-- step 2 --
Thought 수상자를 확인했습니다. 이제 연구 분야를 조회해야 합니다.
Action Search[존 홉필드 제프리 힌턴 연구 분야]
Observation 홉필드 네트워크, 볼츠만 머신, 역전파 알고리즘 등 인공 신경망 개척.

Final Answer: 존 홉필드와 제프리 힌턴, 주요 분야는 인공 신경망과 기계 학습의 기초 이론


#### 실제 AI Agent
사용자 질문 입력
      ↓
LLM이 Thought + Action 텍스트를 생성  ← 여기서 AI가 직접 추론
      ↓
Parser가 텍스트에서 action, action_input 추출
      ↓
실제 Search 도구 실행 (진짜 검색 API 호출)
      ↓
결과(Observation)를 다시 LLM에게 전달
      ↓
LLM이 다시 Thought + Action 생성
      ↓
Finish가 나올 때까지 반복

In [15]:
class ReActPromptTemplate:
    def __init__(self):
        self.tools = []     # 사용 사용 가능한 도구들을 담는 리스트
        self.examples = []  # Few-shot 예시들을 담는 리스트
        self.system_instruction = "" # 에이전트 역할/규칙 설명 문자열

    def set_system_instruction(self, instruction):
        """에이전트의 역할과 행동 규칙을 설정하는 메서드"""
        self.system_instruction = instruction
        return self

    def add_tool(self, name, description, usage_example):
        """
        LLM이 사용할 수 있는 도구를 등록하는 메서드
        nmae : 도구 이름
        description : 도구 설명
        usage_example : 사용법 형식
        """
        self.tools.append({'name': name, 'description': description, 'usage': usage_example})
        return self

    def add_example(self, question, react_trace):
        """
        Few-shot 예시를 추가하는 메서드
        question : 예시 질문
        react_trace : 그 질문에 대한 Thought/Action/Observation 전체 텍스트
        """
        self.examples.append({'question': question, 'react_trace': react_trace})
        return self

    def build(self, user_question):
        """
        (system_instruction, tools, examples)를 조합해서 최종 프롬프트 문자열 하나로 만들어 주는 메서드
        """
        prompt_parts = []   # 프롬프트 조각들을 순서대로 담을 리스트
                            # 마지막에 "\n".join()으로 한 번에 합칠 예정
        # 1. 시스템 지시 추가
        prompt_parts.append(self.system_instruction)    # 역할/규칙 설명
        prompt_parts.append("")

        # 2. 도구 설명 추가
        for tool in self.tools:
            prompt_parts.append(f" -{tool['name']} : {tool['description']}")
            prompt_parts.append(f" 사용법: {tool['usage']}")
        prompt_parts.append("")

        # 3. 출력 형식 명시
        # 나중에 Parser가 이 형식을 기준으로 파싱하기 때문에 매우 중요
        prompt_parts.append("출력 형식:")
        prompt_parts.append("Thought: [현재 상황에 대한 추론]")
        prompt_parts.append("Action: [도구 이름][입력값]")
        prompt_parts.append("Observation: [도구 실행 결과]")
        prompt_parts.append("... (필요한 만큼 반복)")
        prompt_parts.append("Thought: [최종 추론]")
        prompt_parts.append("Action: Finish[최종 답변]")
        prompt_parts.append("")

        # 4. Few-shot 예시 추가
        if self.examples:   # 예시가 하나라도 있을 때만 추가
            prompt_parts.append("예시:")
            for i, ex in enumerate(self.examples, 1):
                prompt_parts.append(f"--- 예시 {i} ---")
                prompt_parts.append(f"Qustion: {ex['question']}")
                prompt_parts.append(ex['react_trace'])
                prompt_parts.append("")

        # 5. 실제 유저 질문 추가
        # 맨 마지막에 붙여야 LLM이 어떤 질문에 답해야 하는지 인식
        prompt_parts.append(f"Question: {user_question}")

        return "\n".join(prompt_parts)

In [16]:
# 실제 사용 예시
template = ReActPromptTemplate()    # 템플릿 객체 생성

template.set_system_instruction(
    # 에이전트의 역할 정의
    "당신은 주어진 질문에 정확하게 답변하기위해 도구를 사용하는 AI 에이전트입니다\n"
    "반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다"
)
# 도구 3개 등록
template.add_tool('Search', '위키피디아에서 정보를 검색합니다.', 'Search[검색어]')
template.add_tool('Lookup', '현재 문서에서 특정 키워드를 찾습니다.', 'Lookup[키워드]')
template.add_tool('Calculator', '수학 계산을 수행합니다.', 'Calculate[수식]')

# Few-shot 예시 등록 (Thought/Action/Observation 전체 흐름을 텍스트로)
example_trace = """Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이면, 인구는 약 210만 명입니다.]
"""

template.add_example("프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?", example_trace)

# build()로 최종 프롬프트 완성 -> 이걸 LLM aPI에 그대로 넘기면 됨
prompt = template.build("독일의 수도는 어디이며, 면적은 얼마인가?")
print(prompt)

당신은 주어진 질문에 정확하게 답변하기위해 도구를 사용하는 AI 에이전트입니다
반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다

 -Search : 위키피디아에서 정보를 검색합니다.
 사용법: Search[검색어]
 -Lookup : 현재 문서에서 특정 키워드를 찾습니다.
 사용법: Lookup[키워드]
 -Calculator : 수학 계산을 수행합니다.
 사용법: Calculate[수식]

출력 형식:
Thought: [현재 상황에 대한 추론]
Action: [도구 이름][입력값]
Observation: [도구 실행 결과]
... (필요한 만큼 반복)
Thought: [최종 추론]
Action: Finish[최종 답변]

예시:
--- 예시 1 ---
Qustion: 프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?
Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이면, 인구는 약 210만 명입니다.]


Question: 독일의 수도는 어디이며, 면적은 얼마인가?


In [ ]:
# ReActParser class
import re
from typing import List, Dict, Optional, Tuple

In [21]:
class ReActParser:
    """
    클래스 변수로 정규표현식 패턴 미리 컴파일
    인스턴스 생성 없이 클래스 자체에 패턴을 저장
    re.compile()로 미리 컴파일해두면 매번 패턴을 새로 해석하지 않아서 빠름
    """
    THOUGHT_PATTERN = re.compile(r'Thought\s*:\s*(.+?)(?=\nAction|$)', re.DOTALL)
    # Thought 뒤의 내용 추출
    # \s* : 콜론 앞 뒤 공배 허용 ("Thought: "처럼 공백 있어도 ok)
    # (.+?) : 실제로 추출할 내용 (괄호 안이 group(1))
    # (?=\nAction|$) : "다음 줄에 Action이 나오거나 텍스트가 끝날 때까지"
    # re.DOTALL: .이 줄바꿈(\n)도 포함해서 매칭 (여러 줄에 걸친 내용도 추출)

    ACTION_PATTERN = re.compile(r'Action\s*:\s*(\w+)\[(.+?)\]', re.DOTALL)
    # "Action: Search[서울 인구]" 에서 "Search"와 "서울 인구"를 각각 추출
    # (\w+)   : 도구 이름 추출 → group(1) (영문자/숫자/언더스코어)
    # \[(.+?)\] : 대괄호 안의 내용 추출 → group(2)
    # .+?     : 최소 매칭 (? 없으면 대괄호를 너무 많이 먹을 수 있음)

    OBSERVATION_PATTERN = re.compile(r'Observation\s*:\s*(.+?)(?=\nThought|$)', re.DOTALL)
    # "Observation:" 뒤의 내용을 추출
    # (?=\nThought|$) : 다음 Thought가 나오거나 텍스트 끝까지만 추출
    
    FINISH_PATTERN = re.compile(r'Action\s*:\s*Finish\[(.+?)\]', re.DOTALL)
    # "Action: Finish[최종답변]" 에서 최종 답변만 추출
    # → 종료 여부 판단 + 최종 답변 추출에 사용
    @classmethod
    def parse_single_step(cls, text: str) -> Dict:
        # @classmethod : 인스턴스 없이 클래스 이름으로 바로 호출 가능
        #                ReActParser.parse_single_step(text)처럼 사용
        # cls : self 대신 클래스 자체를 받음

        result = {"thought": None, "action": None, "action_input": None, "observation": None}
    
        thought_match = cls.THOUGHT_PATTERN.search(text)
        if thought_match:
            result["thought"] = thought_match.group(1).strip()
            # .group(1) : 첫 번째 괄호 안의 내용 = 실제 추론 텍스트
            
        # Action 추출 (도구이름 + 입력값 동시에)
        action_match = cls.ACTION_PATTERN.search(text)
        if action_match:
            result["action"] = action_match.group(1).strip()        # 도구 이름
            result["action_input"] = action_match.group(2).strip()  # 입력값
        
        # Observation 추출
        obs_match = cls.OBSERVATION_PATTERN.search(text)
        if obs_match:
            result["observation"] = obs_match.group(1).strip()
        
        return result  # {"thought": "...", "action": "Search", ...} 형태로 반환
    
    @classmethod
    def parse_full_trace(cls, text: str) -> Dict:
        # 전체 ReAct 텍스트를 받아서 모든 단계를 한 번에 파싱
        steps = []
        final_answer = None
        
        # ── 1. 최종 답변 먼저 추출 ─────────────────────
        finish_match = cls.FINISH_PATTERN.search(text)
        if finish_match:
            final_answer = finish_match.group(1).strip()
        # → 전체 텍스트에서 Finish[...] 를 먼저 찾아두는 이유:
        #   나중에 단계별로 쪼갤 때 Finish 부분도 일반 Action으로 파싱될 수 있어서
        #   미리 따로 빼두는 것
        
        # ── 2. Thought 기준으로 텍스트를 단계별로 분리 ──
        thought_splits = re.split(r'(?=Thought\s*:)', text)
        # (?=Thought\s*:) : "Thought:" 앞에서 쪼개기 (Lookahead)
        #                   단, "Thought:" 자체는 삭제하지 않고 유지
        # → 예시:
        #   "Thought: A\nAction: ...\nThought: B\nAction: ..."
        #    ↓ split 후
        #   ["Thought: A\nAction: ...\n", "Thought: B\nAction: ..."]
        
        thought_splits = [s.strip() for s in thought_splits if s.strip()]
        # 빈 문자열 제거 + 앞뒤 공백 제거
        # → split 결과에 빈 조각이 생길 수 있어서 필터링
        
        # ── 3. 각 조각을 parse_single_step으로 파싱 ────
        for split in thought_splits:
            step = cls.parse_single_step(split)  # 단일 단계 파싱
            if step["thought"]:   # thought가 있는 경우만 유효한 단계로 인정
                steps.append(step)
        
        return {
            "steps": steps,             # 각 단계 딕셔너리 리스트
            "final_answer": final_answer,  # 최종 답변
            "num_steps": len(steps)     # 총 단계 수
        }
    @classmethod
    def is_finished(cls, text: str) -> bool:
        # Finish 패턴이 있는지 없는지만 True/False로 반환
        # → Agent 루프에서 "이제 멈춰야 하나?" 판단할 때 사용
        return bool(cls.FINISH_PATTERN.search(text))
        # bool() : 매칭 객체가 있으면 True, None이면 False

    @classmethod
    def extract_action(cls, text: str) -> Optional[Tuple[str, str]]:
        # Action만 빠르게 추출해서 (도구이름, 입력값) 튜플로 반환
        # Optional : 매칭 안 되면 None 반환 가능
        # Tuple    : ("Search", "서울 인구") 처럼 두 값을 묶어서 반환
        match = cls.ACTION_PATTERN.search(text)
        if match:
            return (match.group(1).strip(), match.group(2).strip())
            # → ("Search", "서울 인구") 형태
        return None  # 매칭 안 되면 None

In [22]:
# ── 에지 케이스 테스트 ──────────────────────────────────────

# Case 1: Thought만 있고 Action이 없는 경우
# → LLM이 아직 Action을 생성 중인 중간 상태일 때
case1 = "Thought: 먼저 검색을 해야 합니다."
result1 = ReActParser.parse_single_step(case1)
# thought="먼저 검색을 해야 합니다.", action=None  ← action이 None이어도 안 터짐

# Case 2: Action이 바로 Finish인 경우
# → 이미 알고 있는 정보라 검색 없이 바로 답변 가능할 때
case2 = "Thought: 이미 알고 있는 정보입니다.\nAction: Finish[지구의 위성은 달입니다.]"
result2 = ReActParser.parse_single_step(case2)
# is_finished()도 True 반환

# Case 3: Action만 빠르게 추출
# → 실제 Agent에서 "어떤 도구를 실행해야 하나?" 판단할 때 자주 씀
case3 = "Thought: 계산이 필요합니다.\nAction: Calculate[15 * 24 + 30]"
action_result = ReActParser.extract_action(case3)
# → ("Calculate", "15 * 24 + 30") 튜플 반환

# Case 4: 완전히 잘못된 형식
# → LLM이 형식을 무시하고 일반 텍스트로 답한 경우
case4 = "이것은 ReAct 형식이 아닌 일반 텍스트입니다."
result4 = ReActParser.parse_single_step(case4)
# → 모든 값이 None, Agent는 이걸 감지하고 재시도하거나 에러 처리